In [1]:
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import platform

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

In [63]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    'latitude': 37.5665,
    'longitude': 126.9780,
    'start_date': '2025-01-01',
    'end_date': '2025-12-31',
    'hourly': 'temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,rain'
}

res = requests.get(url, params = params)
data = res.json()
data

{'latitude': 37.57469,
 'longitude': 126.96,
 'generationtime_ms': 113.43216896057129,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 34.0,
 'hourly_units': {'time': 'iso8601',
  'temperature_2m': '°C',
  'relative_humidity_2m': '%',
  'surface_pressure': 'hPa',
  'wind_speed_10m': 'km/h',
  'rain': 'mm'},
 'hourly': {'time': ['2025-01-01T00:00',
   '2025-01-01T01:00',
   '2025-01-01T02:00',
   '2025-01-01T03:00',
   '2025-01-01T04:00',
   '2025-01-01T05:00',
   '2025-01-01T06:00',
   '2025-01-01T07:00',
   '2025-01-01T08:00',
   '2025-01-01T09:00',
   '2025-01-01T10:00',
   '2025-01-01T11:00',
   '2025-01-01T12:00',
   '2025-01-01T13:00',
   '2025-01-01T14:00',
   '2025-01-01T15:00',
   '2025-01-01T16:00',
   '2025-01-01T17:00',
   '2025-01-01T18:00',
   '2025-01-01T19:00',
   '2025-01-01T20:00',
   '2025-01-01T21:00',
   '2025-01-01T22:00',
   '2025-01-01T23:00',
   '2025-01-02T00:00',
   '2025-01-02T01:00',
   '2025-01-02T02:00',
   '202

In [64]:
# 날씨 데이터들을 따로 추출

df = pd.DataFrame(data['hourly'])
df.head(1)

,time,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,rain
0,2025-01-01T00:00,0.7,65,1018.2,7.3,0.0


In [65]:
df.columns = ['측정 시간', '기온', '상대 습도', '지면 기압', '풍속', '강수량']
df.head(1)

,측정 시간,기온,상대 습도,지면 기압,풍속,강수량
0,2025-01-01T00:00,0.7,65,1018.2,7.3,0.0


In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   측정 시간   8760 non-null   str    
 1   기온      8760 non-null   float64
 2   상대 습도   8760 non-null   int64  
 3   지면 기압   8760 non-null   float64
 4   풍속      8760 non-null   float64
 5   강수량     8760 non-null   float64
dtypes: float64(4), int64(1), str(1)
memory usage: 410.8 KB


#### 연습문제

1. 날씨 데이터에서 분류할 데이터가 존재하지 않는다.
    - 강수량 데이터를 기준으로 0보다 크다면 1(비), 아니라면 0(맑음) 데이터를 이용하여 target column 생성
    - target data의 분포 확인

2. 측정 시간 column을 시계열 데이터로 변경

3. Dataset을 만들어서 구간을 3일치 데이터(72시간)로 잡는다.

4. DataLoader를 생성, batch = 64, shuffle = False

5. LSTM 정의
    - LSTM은 기본값들을 사용, batch_first = True

6. 모델 생성, 손실 함수, 옵티마이저 구성

7. 반복 학습, epochs = 20
    - 반복마다 loss 값 출력
    - 정확도 출력 (후순위)

8. 학습된 모델을 이용하여 예측

**1**

In [67]:
df['강수량'].describe()

count    8760.000000
mean        0.145137
std         0.760927
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        17.700000
Name: 강수량, dtype: float64

In [68]:
for i in range(len(df)):
    if df.iloc[i, 5] != 0.0:
        df.loc[i, 'target'] = 1
    else:
        df.loc[i, 'target'] = 0

In [69]:
df['target'].value_counts()

target
0.0    7506
1.0    1254
Name: count, dtype: int64

**2**

In [80]:
df['측정 시간'] = pd.to_datetime(df['측정 시간'])

In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   측정 시간   8760 non-null   datetime64[us]
 1   기온      8760 non-null   float64       
 2   상대 습도   8760 non-null   int64         
 3   지면 기압   8760 non-null   float64       
 4   풍속      8760 non-null   float64       
 5   강수량     8760 non-null   float64       
 6   target  8760 non-null   float64       
dtypes: datetime64[us](1), float64(5), int64(1)
memory usage: 479.2 KB


**3**

In [166]:
class WindowDS(Dataset):
    def __init__(self, _x, _y, _window):
        self.x = _x
        self.y = _y
        self.window = _window
        self.n = len(_x) - _window
    def __len__(self):
        return max(self.n, 1)
    def __getitem__(self, idx):
        x = self.x[idx : idx + self.window]
        y = self.y[idx + self.window]
        x_tensor = torch.tensor(x, dtype=torch.float32)
        y_tensor = torch.tensor(y.values, dtype = torch.long)
        return x_tensor, y_tensor

In [175]:
X = df[['기온', '상대 습도', '지면 기압', '풍속']].values
y = df['target'].values

In [176]:
scaler_x = MinMaxScaler()

X = scaler_x.fit_transform(X)

In [177]:
weather_ds = WindowDS(X, y, 72)

**4**

In [178]:
train_dl = DataLoader(weather_ds, batch_size=64, shuffle=False)
train_dl

**5**

In [179]:
class WeatherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(WeatherLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.model = nn.Linear(hidden_size, 2),
    
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        pred = self.model(h_n[-1])
        return pred

**6**

In [180]:
model = WeatherLSTM(4)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

In [181]:
model.train()

for epoch in range(20):
    total_loss, total_n = 0.0, 0
    for x, y in train_dl:
        pred = model(x)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        total_n += y.size(0)
        _, pred = torch.max(res, 1)
    train_loss = total_loss / max(total_n, 1)

    if (epoch+1) % 5 == 0:
        print(f"Epoch: {epoch+1} / train_MSE : {round(train_loss, 6)}")

AttributeError: 'numpy.float64' object has no attribute 'values'